# TFM: Parte de recopilación de datos.
### Autores: 
- ####  Adrian
- ####  Carlos Mendívil Gómez
- ####  Guillermo
- ####  Jose



# Introducción.
Para la recopilación de datos se optó por establecer contacto con la empresa Idealista con el fin de obtener acceso a la información de los anuncios publicados en su plataforma. Esta decisión se fundamenta en que la recolección de dichos datos mediante *web scraping* no está permitida según sus términos legales de uso. El procedimiento de extracción se estructurará en dos fases debido al modo de utilización de la API proporcionada por Idealista.
* Parte 1: Uso de la API de Idealista.
* Parte 2: Homogeneización de los datos.


# API de Idealista

Al establecer contacto con los recursos oficiales proporcionados por la página web de Idealista, se nos facilitaron **tres pares de credenciales (API key – secret)**, además de un par adicional destinado exclusivamente a la validación y prueba del código desarrollado.  

Cada uno de estos pares de credenciales presenta limitaciones específicas tanto en número de solicitudes como en el tiempo de uso: el acceso está restringido a **dos meses**, y cada API key dispone de un **máximo de 100 llamadas mensuales**.  

En la práctica, cada llamada a la API generaba un **access token**, el cual debía ser utilizado para realizar las peticiones de datos. Mediante dicho token, era posible acceder como máximo a **50 anuncios por solicitud**, siempre que se aplicara un filtrado adecuado en los parámetros de búsqueda.  

En resumen:  

- **3 API keys**  
  - Límite de **100 llamadas mensuales** cada una  
  - Acceso restringido a **2 meses de uso**  
- **Cada llamada** genera un **access token**  
- **Cada access token** permite obtener hasta **50 anuncios**  

De este modo, realizando una gestión eficiente de las llamadas y aprovechando al máximo las credenciales otorgadas, se podría acceder a un volumen aproximado de **30.000 anuncios de viviendas**.  

$$
\text{Total de anuncios} = \text{nº API-keys} \cdot \text{meses de uso} \cdot \text{access token} \cdot \text{máximo de anuncios por cada access token} \\
\text{Total de anuncios} = 3 \cdot 2 \cdot 100 \cdot 50 = 30000
$$

En la práctica, no fue posible alcanzar el volumen teórico de anuncios previsto, debido a diversas limitaciones encontradas durante el proceso de recopilación de datos, entre ellas:  

* Presencia de anuncios duplicados al aplicar distintos filtros de búsqueda.  
* Publicación repetida de un mismo inmueble por parte de los usuarios de Idealista.  
* Persistencia de ciertos anuncios de un mes a otro, lo que generaba redundancia en los resultados.  

Como consecuencia de estas restricciones, el número final de registros recopilados descendió a poco más de **26.000 anuncios**, todos ellos correspondientes a la ciudad de **Madrid**, con el fin de reducir el sesgo geográfico en el análisis posterior.  


# Uso de la API 
Para poder hacer uso de la API de Idealista fue necesario procesar previamente las **API-keys** y sus correspondientes **secret** proporcionados por la plataforma. En primer lugar, estas credenciales debían ser codificadas mediante el algoritmo *base64*. Una vez codificada la API-key y enviada al enlace de autorización de la API de Idealista, se recibía como respuesta un *JSON* que contenía el **access token**. Se ha creado la función `idealista_token` para realizarlo.

Con este *access token* es posible establecer comunicación directa con la API. Para ello, se empleó la función `query`, a la cual se le proporcionaba tanto el *access token*, además de un filtro diseñado específicamente para detallar los elementos que se deseaba consultar.  



In [1]:
import requests
import base64
from datetime import datetime, timedelta

from dotenv import load_dotenv
import os

import pandas as pd

In [ ]:
#------------------------------------------------------------------------------------------------------------
# Pedir a la API que me devuelva un token 
def idealista_token(api_key, secret):
    """ Obtiene un token de acceso desde la API de Idealista usando OAuth2

    Parameters
    ----------
    api_key(str): Api key
    secret(str): secreto

    Returns
    -------
    access_token, fecha caducidad
    """
    real_credentials = api_key+":"+secret
    base64_bytes = base64.b64encode(real_credentials.encode("ascii"))
    credentials = base64_bytes.decode("ascii")


    req = requests.post(
        "https://api.idealista.com/oauth/token",
        "grant_type=client_credentials&scope=read",
        headers={
            "Authorization": f"Basic {credentials}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
    ).json()
    return req["access_token"], datetime.now() + timedelta(seconds=req["expires_in"])

#--------------------------------------------------------------------------------------------------------------
# Realiza una consulta a la API de idealista
def query(token, request):
    """Realiza una consulta a la API de idealista
    Parameter
    ---------
    token: token recibido por la API de outhenticator
    request: diccionario con los datos de la solicitud

    Returns
    -------
    Devuelve la respuesta de la API
    """

    response = requests.post(
            "https://api.idealista.com/3.5/es/search",
            headers={
                "Authorization": f"Bearer {token}",
                "User-Agent": "curl/8.3.0"
            },
            data=request,
        )
    return response.json()


Es importante señalar que tanto la **API key** como el **secret** son credenciales privadas. Por este motivo, se optó por almacenarlas como **variables de entorno** en un archivo `.env`, lo que permitió garantizar un mayor nivel de seguridad y privacidad en su uso.  

No obstante, esta decisión implicaba que cada vez que se deseaba cambiar de par *API key – secret* era necesario **recargar el entorno completo**, lo cual resultaba en un proceso poco eficiente. Sin embargo, dado que este cambio solo se preveía realizar en contadas ocasiones a lo largo del desarrollo del trabajo, se consideró que esta aproximación era más adecuada que incrementar la complejidad del código para automatizar dicha gestión.  


In [ ]:
# Accedemos a las varibles de entorno y las guardamos en variables locales
load_dotenv()
api_key = os.getenv("api_key")
secret = os.getenv("secret")

La API devolvía como respuesta un archivo en formato *JSON* con los anuncios correspondientes a cada llamada realizada con el *access token*. Dado que el número de *access tokens* era limitado, se optó por almacenar los resultados de cada solicitud en archivos independientes con extensión *.csv*, evitando así el riesgo de perder información por limitaciones de memoria RAM durante el proceso de recopilación.  

In [ ]:
# Hacemos un bucle para acceder en cada token a una página distinta
for num_page in range(201,301):
    # Filtro con los anuncios que queremos recopilar
    filter_request = {
        "country": "es",
        "operation": "sale",
        "propertyType": "homes",
        "locationId": "0-EU-ES-28",
        "maxItems": 50,
        "numPage": num_page
    }
    token, expires = idealista_token(api_key, secret)
    print(token)
    response = query(token, filter_request)

    # Lista de viviendas dentro del JSON
    viviendas = response["elementList"]

    # Usamos json_normalize con sep='.' para que las claves anidadas se aplanen con ese separador
    df = pd.json_normalize(viviendas, sep='.')

    df.to_csv(f"data2/datos_separados/datos_sec_pag_{num_page}.csv", index=False)

## Combinación de todos los csv

Una vez almacenados todos los archivos en formato *.csv*, se procedió a unirlos en un único **DataFrame** con el fin de facilitar su tratamiento posterior.  

Cabe destacar que, al recibir los archivos en formato *JSON* con los detalles de los anuncios, no todos contenían las mismas variables: en muchos casos determinadas columnas se encontraban vacías o directamente no existían en ese archivo concreto. Como consecuencia, al realizar la concatenación de los distintos *.csv*, el **DataFrame** resultante presentó un elevado número de columnas con valores nulos.  

Este inconveniente fue previsto y se resolvió en fases posteriores del proceso de depuración y limpieza de los datos.  

In [5]:
import numpy as np
import pandas as pd

import re
import requests
from io import StringIO

In [8]:
# URLs base
repo_html_url = "https://github.com/guille1006/TFM/tree/main/data2/datos_separados"
raw_base = "https://raw.githubusercontent.com/guille1006/TFM/main/Idealista/datos/"
raw_base = "https://raw.githubusercontent.com/guille1006/TFM/refs/heads/main/data2/datos_separados/"

# Obtener lista de archivos
html = requests.get(repo_html_url).text
csv_files = re.findall(r'datos_sec_pag_\d+\.csv', html)
csv_files = [(int(file.split('_')[3].split('.')[0]), file) for file in set(csv_files)]
csv_files = sorted(csv_files, key=lambda x: x[0])

print(f"Archivos encontrados: {len(csv_files)}")


# Al haber automatizado la obtención de datos, hay algunos .csv que están vacíos
dataframes = dict()
errores = []

for num_page, file in csv_files:
    file_url = raw_base + file

    # Descargaremos toda la información dentro de cada enlace para ver que tiene
    resp = requests.get(file_url)
    content = resp.text.strip()

    # Puede ser que no tengan contenido debido a que se acabaron el tipo de viviendas para el filtro usado
    if not content:
        errores.append((num_page, "Archivo vacío"))
        continue

    # Usaremos StringIO para poder pasar el contenido a un dataframe de pandas
    df = pd.read_csv(StringIO(content))

    # Por motivos de eficiencia de tiempo, hemos decidido ir guardando todos los df en una lista
    # que luego usaremos para concatenar todos ellos
    dataframes[num_page] = df

# Vamos a ordenar el set de all_columns y guardarlo como una lista
dfs = list(dataframes.values())

raw_data = pd.concat(dfs, ignore_index=True, sort=True)

# Tambien eliminaremos las filas duplicadas
initial_rows = raw_data.shape[0]
raw_data = raw_data.drop_duplicates()
final_rows = raw_data.shape[0]

print(f"Teniamos un total de {final_rows-initial_rows} duplicadas")

raw_data.to_csv("data2/raw_data.csv", index=False)

Archivos encontrados: 299
Teniamos un total de -2361 duplicadas


In [10]:
url_1 = "https://raw.githubusercontent.com/guille1006/TFM/refs/heads/main/data/raw_data.csv"
url_2 = "https://raw.githubusercontent.com/guille1006/TFM/refs/heads/main/data2/raw_data.csv"
df_1 = pd.read_csv(url_1)
df_2 = pd.read_csv(url_2)

total_data = pd.concat([df_1, df_2], ignore_index=True, sort=True)

# Tambien eliminaremos las filas duplicadas
initial_rows = total_data.shape[0]
total_data = total_data.drop_duplicates()
final_rows = total_data.shape[0]

print(f"Teniamos un total de {initial_rows-final_rows} duplicadas")
print(f"Y tenemos todas estas filas finales {final_rows}")

total_data.to_csv("data2/total_data.csv", index=False)

Teniamos un total de 3783 duplicadas
Y tenemos todas estas filas finales 26267
